# Explainable Credit Default Prediction for Microfinance
### A LightGBM–SHAP Framework on the Give Me Some Credit Dataset

**Paper contribution:**
1. Rigorous data cleaning of a real-world credit dataset (sentinel values, outliers, missingness)
2. Comparison of three classifiers: Logistic Regression, XGBoost, LightGBM
3. SHAP explainability analysis — feature-level transparency for regulatory compliance
4. Gini coefficient and KS statistic alongside AUROC — the metrics used in practice by credit risk teams
5. Microfinance deployment framing for resource-constrained East African lenders

**Dataset:** Give Me Some Credit (Kaggle) — 150,000 borrowers, 10 features, predict 2-year serious delinquency.  
**Accelerator:** CPU is sufficient — all models are tabular.

In [1]:
import os, warnings, json
import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
warnings.filterwarnings('ignore')

from sklearn.linear_model    import LogisticRegression
from sklearn.preprocessing   import StandardScaler
from sklearn.pipeline        import Pipeline
from sklearn.impute           import SimpleImputer
from sklearn.metrics          import (roc_auc_score, average_precision_score,
                                       roc_curve, precision_recall_curve)
from sklearn.model_selection  import StratifiedKFold
from sklearn.calibration      import calibration_curve
import xgboost  as xgb
import lightgbm as lgb
import shap

DATA_DIR = "/kaggle/input/datasets/carloscrou/give-me-some-credit"
SEED     = 42
TARGET   = 'SeriousDlqin2yrs'
np.random.seed(SEED)

print("Libraries loaded.")
print("LightGBM:", lgb.__version__)
print("XGBoost: ", xgb.__version__)
print("SHAP:    ", shap.__version__)

Libraries loaded.
LightGBM: 4.6.0
XGBoost:  3.2.0
SHAP:     0.51.0


## 1. Load & explore

In [2]:
df = pd.read_csv(f"{DATA_DIR}/cs-training.csv", index_col=0)
print("Shape:", df.shape)
print(f"\nDefault rate: {df[TARGET].mean()*100:.2f}%  "
      f"({df[TARGET].sum():,} defaults / {len(df):,} borrowers)")
print("\nMissing values:")
miss = df.isnull().sum()
print(miss[miss > 0])
print("\nFeature summary:")
df.describe().round(2)

Shape: (150000, 11)

Default rate: 6.68%  (10,026 defaults / 150,000 borrowers)

Missing values:
MonthlyIncome         29731
NumberOfDependents     3924
dtype: int64

Feature summary:


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
count,150000.00,150000.00,150000.00,150000.00,150000.00,120269.00,150000.00,150000.00,150000.00,150000.00,146076.00
mean,0.07,6.05,52.30,0.42,353.01,6670.22,8.45,0.27,1.02,0.24,0.76
std,0.25,249.76,14.77,4.19,2037.82,14384.67,5.15,4.17,1.13,4.16,1.12
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,0.00,0.03,41.00,0.00,0.18,3400.00,5.00,0.00,0.00,0.00,0.00
50%,0.00,0.15,52.00,0.00,0.37,5400.00,8.00,0.00,1.00,0.00,0.00
75%,0.00,0.56,63.00,0.00,0.87,8249.00,11.00,0.00,2.00,0.00,1.00
max,1.00,50708.00,109.00,98.00,329664.00,3008750.00,58.00,98.00,54.00,98.00,20.00


In [3]:
# Class imbalance
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
counts = df[TARGET].value_counts()
axes[0].bar(['Non-default\n(0)', 'Default\n(1)'], counts.values,
            color=['#1A3A5C', '#B91C1C'], alpha=0.85, width=0.5)
axes[0].set_ylabel('Count'); axes[0].set_title('Class Distribution')
axes[0].spines[['top','right']].set_visible(False)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 500, f'{v:,}\n({v/len(df)*100:.1f}%)',
                 ha='center', fontsize=9)

# Missing value heatmap
miss_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
miss_pct = miss_pct[miss_pct > 0]
axes[1].barh(miss_pct.index, miss_pct.values, color='#2563EB', alpha=0.8)
axes[1].set_xlabel('Missing (%)'); axes[1].set_title('Missing Values')
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('fig_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_eda.png")

Saved fig_eda.png


## 2. Data cleaning
Three known issues: (i) sentinel value `98` in delinquency columns, (ii) extreme outliers in utilisation and debt ratio, (iii) single age=0 record.

In [4]:
LATE_COLS = ['NumberOfTime30-59DaysPastDueNotWorse',
             'NumberOfTimes90DaysLate',
             'NumberOfTime60-89DaysPastDueNotWorse']

def clean(df):
    df = df.copy()

    # 1. Sentinel values: 98 in late-payment columns → NaN then cap at 10
    for col in LATE_COLS:
        df[col] = df[col].replace(96, np.nan).replace(98, np.nan)
        df[col] = df[col].clip(upper=10)

    # 2. Invalid age
    df.loc[df['age'] == 0, 'age'] = np.nan

    # 3. Outlier caps (99th percentile for skewed financials)
    df['RevolvingUtilizationOfUnsecuredLines'] = df[
        'RevolvingUtilizationOfUnsecuredLines'].clip(0, 1)
    for col in ['DebtRatio', 'MonthlyIncome']:
        cap = df[col].quantile(0.99)
        df[col] = df[col].clip(upper=cap)

    return df

df_clean = clean(df)
print("Cleaning complete.")
print("Remaining missing values:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

Cleaning complete.
Remaining missing values:
age                                         1
NumberOfTime30-59DaysPastDueNotWorse      269
MonthlyIncome                           29731
NumberOfTimes90DaysLate                   269
NumberOfTime60-89DaysPastDueNotWorse      269
NumberOfDependents                       3924
dtype: int64


## 3. Feature engineering

In [5]:
def engineer(df):
    df = df.copy()
    # Aggregate delinquency signal
    df['MaxDelqStatus']       = df[LATE_COLS].max(axis=1)
    df['TotalDelinquencies']  = df[LATE_COLS].sum(axis=1)
    df['AnyDelinquency']      = (df['TotalDelinquencies'] > 0).astype(int)

    # Income-adjusted features
    df['MonthlyIncome_log']   = np.log1p(df['MonthlyIncome'])
    df['IncomePerDependent']  = df['MonthlyIncome'] / (df['NumberOfDependents'].fillna(0) + 1)

    # Approximate monthly debt burden
    df['EstMonthlyDebt']      = df['DebtRatio'] * df['MonthlyIncome']
    df['EstMonthlyDebt_log']  = np.log1p(df['EstMonthlyDebt'])
    return df

df_feat = engineer(df_clean)

FEATURES = [c for c in df_feat.columns if c != TARGET]
X = df_feat[FEATURES]
y = df_feat[TARGET]

print(f"Features ({len(FEATURES)}):", FEATURES)
print("\nClass balance:", y.value_counts().to_dict())

Features (17): ['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents', 'MaxDelqStatus', 'TotalDelinquencies', 'AnyDelinquency', 'MonthlyIncome_log', 'IncomePerDependent', 'EstMonthlyDebt', 'EstMonthlyDebt_log']

Class balance: {0: 139974, 1: 10026}


## 4. Train / validation split

In [6]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y)

print(f"Train : {len(X_tr):,}  (defaults: {y_tr.sum():,}, {y_tr.mean()*100:.2f}%)")
print(f"Val   : {len(X_val):,}  (defaults: {y_val.sum():,}, {y_val.mean()*100:.2f}%)")

scale_pos = (y_tr == 0).sum() / (y_tr == 1).sum()
print(f"\nscale_pos_weight (class ratio): {scale_pos:.2f}")

Train : 120,000  (defaults: 8,021, 6.68%)
Val   : 30,000  (defaults: 2,005, 6.68%)

scale_pos_weight (class ratio): 13.96


## 5. Models

In [7]:
# ── Logistic Regression baseline ────────────────────────────────────────────
lr_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     LogisticRegression(class_weight='balanced', max_iter=1000,
                                   random_state=SEED, C=0.1))
])
lr_pipe.fit(X_tr, y_tr)
lr_prob = lr_pipe.predict_proba(X_val)[:, 1]
print(f"Logistic Regression  AUROC: {roc_auc_score(y_val, lr_prob):.4f}")

Logistic Regression  AUROC: 0.8620


In [8]:
# ── XGBoost ──────────────────────────────────────────────────────────────────
from sklearn.impute import SimpleImputer
imp = SimpleImputer(strategy='median')
X_tr_imp  = imp.fit_transform(X_tr)
X_val_imp = imp.transform(X_val)

xgb_model = xgb.XGBClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos,
    eval_metric='auc', early_stopping_rounds=30,
    random_state=SEED, verbosity=0
)
xgb_model.fit(X_tr_imp, y_tr,
              eval_set=[(X_val_imp, y_val)],
              verbose=False)
xgb_prob = xgb_model.predict_proba(X_val_imp)[:, 1]
print(f"XGBoost              AUROC: {roc_auc_score(y_val, xgb_prob):.4f}")

XGBoost              AUROC: 0.8677


In [9]:
# ── LightGBM (main model) ────────────────────────────────────────────────────
lgb_model = lgb.LGBMClassifier(
    n_estimators=1000, learning_rate=0.02,
    num_leaves=63, max_depth=-1,
    min_child_samples=50,
    subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    scale_pos_weight=scale_pos,
    random_state=SEED, verbose=-1
)
lgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False),
               lgb.log_evaluation(period=-1)]
)
lgb_prob = lgb_model.predict_proba(X_val)[:, 1]
print(f"LightGBM             AUROC: {roc_auc_score(y_val, lgb_prob):.4f}")
print(f"Best iteration: {lgb_model.best_iteration_}")

LightGBM             AUROC: 0.8634
Best iteration: 8


## 6. Evaluation — AUROC, Gini, KS statistic
Gini and KS are the standard metrics used by credit risk teams in practice.

In [10]:
def gini(y_true, y_prob):
    return 2 * roc_auc_score(y_true, y_prob) - 1

def ks_stat(y_true, y_prob):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    return float(np.max(tpr - fpr))

results = {}
for name, prob in [('Logistic Regression', lr_prob),
                   ('XGBoost',             xgb_prob),
                   ('LightGBM',            lgb_prob)]:
    results[name] = {
        'AUROC': round(roc_auc_score(y_val, prob), 4),
        'Gini':  round(gini(y_val, prob), 4),
        'KS':    round(ks_stat(y_val, prob), 4),
        'AUPRC': round(average_precision_score(y_val, prob), 4),
    }
    print(f"{name:<22} AUROC={results[name]['AUROC']:.4f}  "
          f"Gini={results[name]['Gini']:.4f}  KS={results[name]['KS']:.4f}  "
          f"AUPRC={results[name]['AUPRC']:.4f}")

json.dump(results, open('model_results.json', 'w'), indent=2)

Logistic Regression    AUROC=0.8620  Gini=0.7239  KS=0.5724  AUPRC=0.3881
XGBoost                AUROC=0.8677  Gini=0.7354  KS=0.5806  AUPRC=0.3979
LightGBM               AUROC=0.8634  Gini=0.7269  KS=0.5783  AUPRC=0.3767


In [11]:
# ROC curves comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
colors = {'Logistic Regression': '#566B7E',
          'XGBoost':             '#2563EB',
          'LightGBM':            '#1A3A5C'}
probs  = {'Logistic Regression': lr_prob,
          'XGBoost':             xgb_prob,
          'LightGBM':            lgb_prob}
for name, prob in probs.items():
    fpr, tpr, _ = roc_curve(y_val, prob)
    auc = roc_auc_score(y_val, prob)
    ax1.plot(fpr, tpr, color=colors[name], lw=2,
             label=f"{name} ({auc:.4f})")
ax1.plot([0,1],[0,1],'--',color='0.7',lw=1)
ax1.set_xlabel('False positive rate'); ax1.set_ylabel('True positive rate')
ax1.set_title('ROC Curves'); ax1.legend(frameon=False, fontsize=9)
ax1.spines[['top','right']].set_visible(False)

# Precision-recall curves
for name, prob in probs.items():
    pr, rc, _ = precision_recall_curve(y_val, prob)
    ap = average_precision_score(y_val, prob)
    ax2.plot(rc, pr, color=colors[name], lw=2,
             label=f"{name} (AP={ap:.4f})")
ax2.axhline(y_val.mean(), color='0.7', lw=1, ls='--',
            label=f"Baseline ({y_val.mean():.3f})")
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
ax2.set_title('Precision–Recall Curves')
ax2.legend(frameon=False, fontsize=9)
ax2.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('fig_roc_pr.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_roc_pr.png")

Saved fig_roc_pr.png


## 7. SHAP explainability — the core contribution
SHAP (SHapley Additive exPlanations) provides borrower-level and population-level explanations. This is what regulators require for fair lending compliance.

In [12]:
# SHAP values for LightGBM
print("Computing SHAP values …")
explainer  = shap.TreeExplainer(lgb_model)
shap_vals  = explainer.shap_values(X_val)

# For binary classification LightGBM returns a list; take class-1
if isinstance(shap_vals, list):
    sv = shap_vals[1]
else:
    sv = shap_vals

print(f"SHAP matrix shape: {sv.shape}")
print("\nMean |SHAP| per feature (global importance):")
imp = pd.Series(np.abs(sv).mean(axis=0), index=X_val.columns)
print(imp.sort_values(ascending=False).round(5))

Computing SHAP values …
SHAP matrix shape: (30000, 17)

Mean |SHAP| per feature (global importance):
RevolvingUtilizationOfUnsecuredLines    0.19836
MaxDelqStatus                           0.13105
age                                     0.04305
MonthlyIncome                           0.02983
EstMonthlyDebt                          0.02813
TotalDelinquencies                      0.02782
AnyDelinquency                          0.02569
NumberOfOpenCreditLinesAndLoans         0.01785
DebtRatio                               0.01427
NumberOfTimes90DaysLate                 0.01008
NumberRealEstateLoansOrLines            0.00963
NumberOfTime30-59DaysPastDueNotWorse    0.00876
IncomePerDependent                      0.00827
EstMonthlyDebt_log                      0.00512
NumberOfTime60-89DaysPastDueNotWorse    0.00474
NumberOfDependents                      0.00336
MonthlyIncome_log                       0.00000
dtype: float64


In [13]:
# ── Figure: SHAP summary (beeswarm) ─────────────────────────────────────────
plt.figure(figsize=(9, 6))
shap.summary_plot(sv, X_val, plot_type='dot', max_display=15,
                  show=False, color_bar=True)
plt.title('SHAP Feature Importance — LightGBM Credit Default Model', pad=12)
plt.tight_layout()
plt.savefig('fig_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_shap_beeswarm.png")

Saved fig_shap_beeswarm.png


In [14]:
# ── Figure: SHAP bar (mean |SHAP|) ──────────────────────────────────────────
plt.figure(figsize=(8, 5))
shap.summary_plot(sv, X_val, plot_type='bar', max_display=12, show=False)
plt.title('Mean |SHAP| — Global Feature Importance', pad=12)
plt.tight_layout()
plt.savefig('fig_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_shap_bar.png")

Saved fig_shap_bar.png


In [15]:
# ── Figure: SHAP dependence plots for top 3 features ────────────────────────
imp_sorted = imp.sort_values(ascending=False)
top3 = imp_sorted.index[:3].tolist()

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for i, feat in enumerate(top3):
    feat_idx = list(X_val.columns).index(feat)
    shap.dependence_plot(feat, sv, X_val, ax=axes[i], show=False,
                         dot_size=8, alpha=0.4)
    axes[i].set_title(feat, fontsize=9.5)
    axes[i].spines[['top','right']].set_visible(False)
plt.suptitle('SHAP Dependence Plots — Top 3 Features', y=1.02, fontsize=11)
plt.tight_layout()
plt.savefig('fig_shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_shap_dependence.png")

Saved fig_shap_dependence.png


## 8. Model calibration
A well-calibrated model outputs probabilities that match real default rates — essential for setting loan approval thresholds.

In [16]:
fig, ax = plt.subplots(figsize=(6, 5))
for name, prob in probs.items():
    frac_pos, mean_pred = calibration_curve(y_val, prob, n_bins=10)
    ax.plot(mean_pred, frac_pos, 's-', color=colors[name],
            lw=1.5, ms=5, label=name)
ax.plot([0,1],[0,1],'--',color='0.6',lw=1,label='Perfect calibration')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives')
ax.set_title('Calibration Curves')
ax.legend(frameon=False, fontsize=9)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('fig_calibration.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_calibration.png")

Saved fig_calibration.png


## 9. Results summary

In [17]:
summary = pd.DataFrame(results).T.reset_index()
summary.columns = ['Model', 'AUROC', 'Gini', 'KS', 'AUPRC']
summary = summary.sort_values('AUROC', ascending=False).reset_index(drop=True)
print("\n── Final Results Table ──────────────────────────────────────────────────")
print(summary.to_string(index=False))
print(f"\nLightGBM best iteration : {lgb_model.best_iteration_}")
print(f"Val set size            : {len(y_val):,}  (default rate {y_val.mean()*100:.2f}%)")
summary.to_csv('results_summary.csv', index=False)
print("\nSaved: results_summary.csv")


── Final Results Table ──────────────────────────────────────────────────
              Model  AUROC   Gini     KS  AUPRC
            XGBoost 0.8677 0.7354 0.5806 0.3979
           LightGBM 0.8634 0.7269 0.5783 0.3767
Logistic Regression 0.8620 0.7239 0.5724 0.3881

LightGBM best iteration : 8
Val set size            : 30,000  (default rate 6.68%)

Saved: results_summary.csv


In [18]:
# Save models for paper artifacts
import pickle
lgb_model.booster_.save_model('lgb_credit_model.txt')
with open('shap_values.pkl', 'wb') as f:
    pickle.dump({'shap_values': sv, 'feature_names': list(X_val.columns)}, f)
print("Saved: lgb_credit_model.txt  shap_values.pkl")
print("\nFigures generated:")
for f in ['fig_eda.png','fig_roc_pr.png','fig_shap_beeswarm.png',
          'fig_shap_bar.png','fig_shap_dependence.png','fig_calibration.png']:
    print(" ", f)

Saved: lgb_credit_model.txt  shap_values.pkl

Figures generated:
  fig_eda.png
  fig_roc_pr.png
  fig_shap_beeswarm.png
  fig_shap_bar.png
  fig_shap_dependence.png
  fig_calibration.png


## 10. Paper notes
**Title:** *Explainable Credit Default Prediction for Microfinance: A LightGBM–SHAP Framework*

**Key results to report:**
- LightGBM macro AUROC, Gini coefficient, KS statistic vs baselines
- Top SHAP features (revolving utilisation, late payment history, age expected to dominate)
- Calibration analysis — is LightGBM better calibrated than LR?

**Contribution framing:**
Credit models in East African microfinance are typically black-box scorecards. SHAP provides:
(1) regulatory-compliant feature attribution (fair lending)
(2) actionable borrower-level explanations ("your revolving utilisation is driving your risk score")
(3) domain validation — if SHAP top features don't match credit theory, the model is suspect

**Comparison benchmark:** Kaggle leaderboard top AUROC was ~0.869; a well-tuned LightGBM should reach 0.860+.